In [ ]:
"""
Project: Hybrid Complementarity Visualizer
Description: Spatial visualization of SRCC, CVI, and Complementarity Clusters 
             with high-precision GridSpec layout.
"""

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pyproj
import os
import warnings

warnings.filterwarnings("ignore")

class LocalComplementarityVisualizer:
    def __init__(self):
        self.input_dir = "../data/processed"
        self.output_dir = "../data/outputs"
        os.makedirs(self.output_dir, exist_ok=True)
        
        self.scenarios = ['hist', 'ssp126', 'ssp245', 'ssp370', 'ssp585']
        self.seasons = ['DJF', 'MAM', 'JJA', 'SON']
        self.extent = [94, 142, -12, 9]

        self.clon, self.clat = 120.0, -2.5
        proj_std = pyproj.Proj(f"+proj=merc +lon_0={self.clon} +lat_ts={self.clat}")
        _, y_center = proj_std(self.clon, self.clat)
        self.data_crs = ccrs.Mercator(central_longitude=self.clon, latitude_true_scale=self.clat, false_northing=-y_center)

        # PRIMARY METRICS
        self.metrics_config = {
            'SRCC':       {'title': 'SRCC', 'vmin': -1.0, 'vmax': 1.0, 'cmap': 'coolwarm', 'extend': 'neither'},
            'CVI':        {'title': 'CVI', 'vmin': 0.5, 'vmax': 1.0, 'cmap': 'RdBu_r', 'extend': 'neither'},
            'SRCC_siang': {'title': 'Spearman Rank Corr (Daylight)', 'vmin': -1.0, 'vmax': 1.0, 'cmap': 'coolwarm', 'extend': 'neither'},
            'CVI_siang':  {'title': 'Composite Var Index (Daylight)', 'vmin': 0.5, 'vmax': 1.0, 'cmap': 'RdBu_r', 'extend': 'neither'}
        }

        # DISCRETE CLUSTER CONFIGURATION (4 CLASSES)
        self.class_colors = ['#d9e8c0', '#fffdda', '#fbb48b', '#f57e79']
        self.class_cmap = mcolors.ListedColormap(self.class_colors)
        self.class_bounds = [0.5, 1.5, 2.5, 3.5, 4.5]
        self.class_norm = mcolors.BoundaryNorm(self.class_bounds, self.class_cmap.N)
        self.class_labels = ['1', '2', '3', '4']

    def load_local_nc(self, metric_type, grid_type, scen):
        fpath = os.path.join(self.input_dir, f"{metric_type}_{grid_type}_{scen}.nc")
        if not os.path.exists(fpath): return None
        with xr.open_dataset(fpath) as ds: return ds[list(ds.data_vars)[0]].load()

    def prep_map(self, ax):
        ax.set_extent(self.extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.7, edgecolor='black', zorder=2)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='dimgray', linestyle='--', zorder=2)
        ax.spines['geo'].set_linewidth(0.5)

    def add_label_box(self, ax, text, loc='bottom_left'):
        kwargs = dict(transform=ax.transAxes, fontsize=12, fontweight='bold', 
                      bbox=dict(boxstyle='square,pad=0.2', facecolor='white', alpha=0.85, edgecolor='gray', zorder=5))
        if loc == 'bottom_right':
            ax.text(0.96, 0.04, text, ha='right', va='bottom', **kwargs)
        elif loc == 'bottom_left':
            ax.text(0.02, 0.05, text, ha='left', va='bottom', **kwargs)

    def format_seasonal_layout(self, ax, i, j, season, top_label):
        if j == 0:
            ax.text(-0.06, 0.5, season, va='center', ha='center', rotation=90, transform=ax.transAxes, fontsize=12, fontweight='bold')
        if i == 0 and top_label:
            ax.set_title(top_label, fontsize=12, fontweight='bold', pad=8)

    def _add_continuous_colorbar(self, fig, cfg, ax_rect):
        cbar_ax = fig.add_axes(ax_rect)
        sm = plt.cm.ScalarMappable(cmap=cfg['cmap'], norm=plt.Normalize(vmin=cfg['vmin'], vmax=cfg['vmax']))
        cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', extend=cfg['extend'])
        cbar.set_label(f"{cfg['title']}", fontsize=12, fontweight='bold')
        cbar.ax.tick_params(labelsize=12)

    def _add_discrete_colorbar(self, fig, ax_rect, mode_label=""):
        cbar_ax = fig.add_axes(ax_rect)
        sm = plt.cm.ScalarMappable(cmap=self.class_cmap, norm=self.class_norm)
        cb = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', ticks=[1, 2, 3, 4])
        cb.ax.set_xticklabels(self.class_labels, fontsize=12, fontweight='bold')
        cb.ax.tick_params(size=0)
        
        title = f"Complementarity Class ({mode_label})" if mode_label else "Complementarity Class"
        cb.set_label(title, fontsize=12, fontweight='bold')

    def plot_mega_seasonal(self, metric_type):
        print(f"   [INFO] Generating Seasonal plot for {metric_type}...")
        fig, axes = plt.subplots(4, 5, figsize=(20, 8), subplot_kw={'projection': ccrs.PlateCarree()})
        cfg = self.metrics_config[metric_type]
        for i, season in enumerate(self.seasons):
            for j, scen in enumerate(self.scenarios):
                ax = axes[i, j]
                self.prep_map(ax)
                da = self.load_local_nc(metric_type, 'Season', scen)
                if da is not None:
                    da.sel(season=season).plot.imshow(x='lon', y='lat', ax=ax, transform=self.data_crs, 
                                                      vmin=cfg['vmin'], vmax=cfg['vmax'], cmap=cfg['cmap'], 
                                                      add_colorbar=False, interpolation='bilinear', zorder=1, add_labels=False)
                self.format_seasonal_layout(ax, i, j, season, scen.upper())

        plt.subplots_adjust(wspace=0.01, hspace=-0.05, bottom=0.08, top=0.92, left=0.05, right=0.98)
        self._add_continuous_colorbar(fig, cfg, [0.05, 0.035, 0.93, 0.03])
        plt.savefig(os.path.join(self.output_dir, f"GRID_Seasonal_{metric_type}.png"), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()

    def plot_mega_climatology(self, metric_type):
        print(f"   [INFO] Generating Climatology plot for {metric_type}...")
        
        fig = plt.figure(figsize=(10, 11))
        gs = fig.add_gridspec(3, 2, height_ratios=[1.6, 1, 1])
        cfg = self.metrics_config[metric_type]

        axes_dict = {
            'hist':   fig.add_subplot(gs[0, :], projection=ccrs.PlateCarree()),
            'ssp126': fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree()),
            'ssp245': fig.add_subplot(gs[1, 1], projection=ccrs.PlateCarree()),
            'ssp370': fig.add_subplot(gs[2, 0], projection=ccrs.PlateCarree()),
            'ssp585': fig.add_subplot(gs[2, 1], projection=ccrs.PlateCarree())
        }

        for scen in self.scenarios:
            ax = axes_dict[scen]
            self.prep_map(ax)
            da = self.load_local_nc(metric_type, 'Clim', scen)
            if da is not None:
                da.plot.imshow(x='lon', y='lat', ax=ax, transform=self.data_crs, 
                               vmin=cfg['vmin'], vmax=cfg['vmax'], cmap=cfg['cmap'], 
                               add_colorbar=False, interpolation='bilinear', zorder=1, add_labels=False)
            self.add_label_box(ax, scen.upper(), loc='bottom_left')

        plt.subplots_adjust(wspace=0.01, hspace=-0.34, bottom=0.10, top=0.97, left=0.04, right=0.96)
        
        pos_left = axes_dict['ssp370'].get_position()
        pos_right = axes_dict['ssp585'].get_position()
        cbar_rect = [pos_left.x0, 0.12, pos_right.x1 - pos_left.x0, 0.022]
        
        self._add_continuous_colorbar(fig, cfg, cbar_rect)
        plt.savefig(os.path.join(self.output_dir, f"GRID_Climatology_{metric_type}.png"), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()

    def _generate_class_data(self, da_srcc, da_cvi):
        if da_srcc is None or da_cvi is None: return None
        da_cls = xr.zeros_like(da_srcc)
        da_cls = xr.where((da_srcc > 0.6) | (da_cvi > 0.9), 4, da_cls)
        da_cls = xr.where((da_cls == 0) & ((da_srcc > 0) | (da_cvi > 0.8)), 3, da_cls)
        da_cls = xr.where((da_cls == 0) & ((da_srcc > -0.6) | (da_cvi > 0.6)), 2, da_cls)
        da_cls = xr.where((da_cls == 0), 1, da_cls)
        return xr.where(da_srcc.isnull(), np.nan, da_cls)

    def plot_class_seasonal(self, mode=''):
        mode_label = "24H" if mode == '' else "Daylight"
        print(f"   [INFO] Generating Seasonal plot for Complementarity CLASS ({mode_label})...")
        fig, axes = plt.subplots(4, 5, figsize=(20, 8), subplot_kw={'projection': ccrs.PlateCarree()})
        for i, season in enumerate(self.seasons):
            for j, scen in enumerate(self.scenarios):
                ax = axes[i, j]
                self.prep_map(ax)
                
                da_srcc = self.load_local_nc(f'SRCC{mode}', 'Season', scen)
                da_cvi = self.load_local_nc(f'CVI{mode}', 'Season', scen)
                
                da_cls = self._generate_class_data(da_srcc, da_cvi)
                if da_cls is not None:
                    da_cls.sel(season=season).plot.imshow(x='lon', y='lat', ax=ax, transform=self.data_crs, 
                                                          cmap=self.class_cmap, norm=self.class_norm, 
                                                          add_colorbar=False, interpolation='nearest', zorder=1, add_labels=False)
                self.format_seasonal_layout(ax, i, j, season, scen.upper())

        plt.subplots_adjust(wspace=0.01, hspace=-0.05, bottom=0.08, top=0.92, left=0.05, right=0.98)
        self._add_discrete_colorbar(fig, [0.05, 0.035, 0.93, 0.03], mode_label=mode_label)
        plt.savefig(os.path.join(self.output_dir, f"GRID_Seasonal_CLASS{mode}.png"), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()

    def plot_class_climatology(self, mode=''):
        mode_label = "24H" if mode == '' else "Daylight"
        print(f"   [INFO] Generating Climatology plot for Complementarity CLASS ({mode_label})...")
        
        fig = plt.figure(figsize=(10, 11))
        gs = fig.add_gridspec(3, 2, height_ratios=[1.6, 1, 1])

        axes_dict = {
            'hist':   fig.add_subplot(gs[0, :], projection=ccrs.PlateCarree()),
            'ssp126': fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree()),
            'ssp245': fig.add_subplot(gs[1, 1], projection=ccrs.PlateCarree()),
            'ssp370': fig.add_subplot(gs[2, 0], projection=ccrs.PlateCarree()),
            'ssp585': fig.add_subplot(gs[2, 1], projection=ccrs.PlateCarree())
        }

        for scen in self.scenarios:
            ax = axes_dict[scen]
            self.prep_map(ax)
            
            da_srcc = self.load_local_nc(f'SRCC{mode}', 'Clim', scen)
            da_cvi = self.load_local_nc(f'CVI{mode}', 'Clim', scen)
            
            da_cls = self._generate_class_data(da_srcc, da_cvi)
            if da_cls is not None:
                da_cls.plot.imshow(x='lon', y='lat', ax=ax, transform=self.data_crs, 
                                   cmap=self.class_cmap, norm=self.class_norm, 
                                   add_colorbar=False, interpolation='nearest', zorder=1, add_labels=False)
            
            self.add_label_box(ax, scen.upper(), loc='bottom_left')

        plt.subplots_adjust(wspace=0.01, hspace=-0.34, bottom=0.10, top=0.97, left=0.04, right=0.96)
        
        pos_left = axes_dict['ssp370'].get_position()
        pos_right = axes_dict['ssp585'].get_position()
        cbar_rect = [pos_left.x0, 0.12, pos_right.x1 - pos_left.x0, 0.022]

        self._add_discrete_colorbar(fig, cbar_rect, mode_label=mode_label)
        plt.savefig(os.path.join(self.output_dir, f"GRID_Climatology_CLASS{mode}.png"), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()

    def run_pipeline(self):
        print("[INFO] INITIALIZING COMPLEMENTARITY VISUALIZATION PIPELINE...")
        
        # 1. Plot SRCC and CVI (24H and Daylight)
        for metric in self.metrics_config.keys():
            print(f"\n{'='*50}\nProcessing: {metric}\n{'='*50}")
            self.plot_mega_seasonal(metric)
            self.plot_mega_climatology(metric)
            
        # 2. Plot Complementarity Class 
        for mode in ['', '_siang']:
            print(f"\n{'='*50}\nProcessing: CLUSTERING CLASS {'(Daylight)' if mode else '(24H)'}\n{'='*50}")
            self.plot_class_seasonal(mode=mode)
            self.plot_class_climatology(mode=mode)
            
        print("\n[SUCCESS] All complementarity maps successfully rendered!")

if __name__ == "__main__":
    LocalComplementarityVisualizer().run_pipeline()

In [ ]:
"""
Project: K-Means Clustering Visualizer & Bappenas Table Extractor
Description: Reads pre-computed CLASS_KMEANS files, renders 2-Class spatial maps 
             with precision GridSpec layout, and extracts regional percentages 
             into a Bappenas Excel table (28 Rows x 7 Columns).
"""

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.path as mpath
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pyproj
import os
import warnings

warnings.filterwarnings("ignore")

class SovereignKMeansVisualizer:
    def __init__(self):
        self.input_dir = "../data/processed"
        self.output_dir = "../data/outputs"
        self.tables_dir = os.path.join(self.output_dir, "tables")
        
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.tables_dir, exist_ok=True)
        
        self.scenarios = ['hist', 'ssp126', 'ssp245', 'ssp370', 'ssp585']
        self.seasons = ['DJF', 'MAM', 'JJA', 'SON']
        self.extent = [94, 142, -12, 9]

        self.clon, self.clat = 120.0, -2.5
        proj_std = pyproj.Proj(f"+proj=merc +lon_0={self.clon} +lat_ts={self.clat}")
        _, y_center = proj_std(self.clon, self.clat)
        self.data_crs = ccrs.Mercator(central_longitude=self.clon, latitude_true_scale=self.clat, false_northing=-y_center)

        # K-MEANS 2-CLASS CONFIGURATION
        self.class_colors = ['#d9e8c0', '#f57e79'] # Soft Green (Ideal) & Soft Red (Risk)
        self.class_cmap = mcolors.ListedColormap(self.class_colors)
        self.class_bounds = [0.5, 1.5, 2.5]
        self.class_norm = mcolors.BoundaryNorm(self.class_bounds, self.class_cmap.N)
        self.class_labels = ['1', '2']
        self.kelas_dict = {1: '1', 2: '2'}

        # BAPPENAS REGIONAL POLYGONS
        self.bappenas_polygons = {
            'Wilayah Sumatera': [(95.040924, 5.902583), (94.096769, 3.707855), (102.600187, -6.492897), (105.495271, -6.237656), (106.254287, -5.446087), (106.419082, -3.370451), (108.502412, -3.400257), (108.520137, -2.436121), (106.707393, -2.139727), (104.756358, 1.281078), (104.052608, 1.239581), (103.562343, 1.040493), (103.403042, 1.192899), (101.667202, 2.167505), (97.442959, 5.488466), (95.542324, 6.111478)],
            'Wilayah Jawa': [(114.410778, -8.200808), (114.412149, -8.137591), (114.431380, -8.081499), (114.493862, -8.041041), (114.921707, -7.376308), (114.092167, -6.782111), (110.784757, -6.312944), (106.092962, -5.749576), (105.784264, -6.110246), (105.363971, -6.435131), (105.020603, -6.577018), (105.221877, -6.948989), (105.544403, -7.271744), (107.533530, -7.829531), (109.914687, -8.142998), (111.916548, -8.569963), (114.644931, -8.862816), (114.593916, -8.504145), (114.430713, -8.378102)],
            'Wilayah Kalimantan': [(109.618594, 2.338713), (108.333194, 1.141777), (108.929341, -3.000747), (114.356587, -4.863942), (117.674458, -3.943826), (119.717915, 2.512211), (118.968213, 4.334317), (115.606397, 4.706689), (114.068311, 2.711468)],
            'Wilayah Bali-Nusra': [(122.718161, -11.095435), (124.563864, -10.966035), (125.486715, -9.690682), (125.212057, -7.801379), (117.741354, -7.801379), (114.665182, -7.997256), (114.432366, -8.089387), (114.415887, -8.193385), (114.437516, -8.234710), (114.483178, -8.397502), (114.741164, -8.590424), (114.888793, -8.969760)],
            'Wilayah Sulawesi': [(120.952826, -7.595560), (123.314887, -7.432181), (124.534369, -5.806185), (123.897162, -0.421857), (125.149604, 0.413095), (126.489936, 1.621378), (123.545600, 2.038645), (119.832221, 1.687269), (117.986518, -4.022070), (118.623725, -6.494336)],
            'Wilayah Maluku': [(128.581218, 2.922818), (127.284831, 2.000828), (124.186686, -1.470282), (125.658854, -8.209784), (131.613444, -8.601042), (135.282878, -6.902902), (135.063151, -5.329928), (132.624186, -4.979787), (131.591472, -4.454232), (130.097331, -0.899118), (129.416179, 0.858625)],
            'Wilayah Papua': [(129.965495, -0.152084), (130.844401, -2.216961), (132.492350, -4.607559), (136.579265, -5.220529), (137.568034, -9.035300), (141.193522, -9.577384), (141.149577, -1.580107), (130.734538, 0.551031)]
        }

    def load_kmeans_nc(self, mode, agg, scen):
        fpath = os.path.join(self.input_dir, f"CLASS_KMEANS{mode}_{agg}_{scen}.nc")
        if not os.path.exists(fpath): 
            return None
        with xr.open_dataset(fpath) as ds: 
            return ds[list(ds.data_vars)[0]].load()

    def prep_map(self, ax):
        ax.set_extent(self.extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.7, edgecolor='black', zorder=2)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='dimgray', linestyle='--', zorder=2)
        ax.spines['geo'].set_linewidth(0.5)

    def add_label_box(self, ax, text, loc='bottom_left'):
        kwargs = dict(transform=ax.transAxes, fontsize=12, fontweight='bold', 
                      bbox=dict(boxstyle='square,pad=0.2', facecolor='white', alpha=0.85, edgecolor='gray', zorder=5))
        if loc == 'bottom_right':
            ax.text(0.96, 0.04, text, ha='right', va='bottom', **kwargs)
        elif loc == 'bottom_left':
            ax.text(0.02, 0.05, text, ha='left', va='bottom', **kwargs)

    def _add_discrete_colorbar(self, fig, ax_rect, mode_label=""):
        cbar_ax = fig.add_axes(ax_rect)
        sm = plt.cm.ScalarMappable(cmap=self.class_cmap, norm=self.class_norm)
        cb = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', ticks=[1, 2])
        cb.ax.set_xticklabels(self.class_labels, fontsize=12, fontweight='bold')
        cb.ax.tick_params(size=0)
        title = f"Complementarity Class ({mode_label})" if mode_label else "Complementarity Class"
        cb.set_label(title, fontsize=12, fontweight='bold')

    # =========================================================================
    # BLOCK 1: MAP RENDERING
    # =========================================================================
    def plot_kmeans_climatology(self, mode=''):
        mode_label = "Daylight" if mode == '_siang' else ""
        print(f"   [INFO] Generating K-Means Climatology Map ({mode_label})...")
        
        fig = plt.figure(figsize=(10, 11))
        gs = fig.add_gridspec(3, 2, height_ratios=[1.6, 1, 1])

        axes_dict = {
            'hist':   fig.add_subplot(gs[0, :], projection=ccrs.PlateCarree()),
            'ssp126': fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree()),
            'ssp245': fig.add_subplot(gs[1, 1], projection=ccrs.PlateCarree()),
            'ssp370': fig.add_subplot(gs[2, 0], projection=ccrs.PlateCarree()),
            'ssp585': fig.add_subplot(gs[2, 1], projection=ccrs.PlateCarree())
        }

        for scen in self.scenarios:
            ax = axes_dict[scen]
            self.prep_map(ax)
            da_cls = self.load_kmeans_nc(mode, 'Clim', scen)
            
            if da_cls is not None:
                da_cls.plot.imshow(x='lon', y='lat', ax=ax, transform=self.data_crs, 
                                   cmap=self.class_cmap, norm=self.class_norm, 
                                   add_colorbar=False, interpolation='nearest', zorder=1, add_labels=False)
            
            self.add_label_box(ax, scen.upper(), loc='bottom_left')

        plt.subplots_adjust(wspace=0.01, hspace=-0.34, bottom=0.10, top=0.97, left=0.04, right=0.96)
        
        pos_left = axes_dict['ssp370'].get_position()
        pos_right = axes_dict['ssp585'].get_position()
        cbar_rect = [pos_left.x0, 0.12, pos_right.x1 - pos_left.x0, 0.022]

        self._add_discrete_colorbar(fig, cbar_rect, mode_label=mode_label)
        
        plt.savefig(os.path.join(self.output_dir, f"GRID_KMEANS_Climatology{mode}.jpg"), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()

    def plot_kmeans_seasonal(self, mode=''):
        mode_label = "Daylight" if mode == '_siang' else "24H"
        print(f"   [INFO] Generating K-Means Seasonal Map ({mode_label})...")
        
        fig, axes = plt.subplots(4, 5, figsize=(20, 8), subplot_kw={'projection': ccrs.PlateCarree()})
        for i, season in enumerate(self.seasons):
            for j, scen in enumerate(self.scenarios):
                ax = axes[i, j]
                self.prep_map(ax)
                
                da_cls = self.load_kmeans_nc(mode, 'Season', scen)
                if da_cls is not None:
                    da_season = da_cls.sel(season=season) if 'season' in da_cls.dims else da_cls.isel(season=i)
                    da_season.plot.imshow(x='lon', y='lat', ax=ax, transform=self.data_crs, 
                                          cmap=self.class_cmap, norm=self.class_norm, 
                                          add_colorbar=False, interpolation='nearest', zorder=1, add_labels=False)
                
                if j == 0:
                    ax.text(-0.06, 0.5, season, va='center', ha='center', rotation=90, transform=ax.transAxes, fontsize=12, fontweight='bold')
                if i == 0:
                    ax.set_title(scen.upper(), fontsize=12, fontweight='bold', pad=8)

        plt.subplots_adjust(wspace=0.01, hspace=-0.05, bottom=0.08, top=0.92, left=0.05, right=0.98)
        self._add_discrete_colorbar(fig, [0.05, 0.035, 0.93, 0.03], mode_label=mode_label)
        plt.savefig(os.path.join(self.output_dir, f"GRID_KMEANS_Seasonal{mode}.jpg"), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()

    # =========================================================================
    # BLOCK 2: BAPPENAS TABLE EXTRACTION
    # =========================================================================
    def _get_polygon_mask(self, da, polygon_coords_degrees):
        lon_grid, lat_grid = np.meshgrid(da.lon.values, da.lat.values)
        points_mercator = np.vstack((lon_grid.flatten(), lat_grid.flatten())).T
        
        poly_lons = np.array([p[0] for p in polygon_coords_degrees])
        poly_lats = np.array([p[1] for p in polygon_coords_degrees])
        
        transformed_poly = self.data_crs.transform_points(ccrs.PlateCarree(), poly_lons, poly_lats)
        poly_x_y = transformed_poly[:, :2] 
        
        path = mpath.Path(poly_x_y)
        mask_flat = path.contains_points(points_mercator)
        mask_2d = mask_flat.reshape(lon_grid.shape)
        
        return xr.DataArray(mask_2d, coords={'lat': da.lat, 'lon': da.lon}, dims=['lat', 'lon'])

    def extract_bappenas_tables(self, mode=''):
        mode_label = "Daylight" if mode == '_siang' else "24H"
        print(f"   [INFO] Extracting Bappenas Regional Table: {mode_label}...")
        
        table_data = []
        class_maps = {}
        
        for scen in self.scenarios:
            da_cls = self.load_kmeans_nc(mode, 'Clim', scen)
            if da_cls is not None:
                class_maps[scen] = da_cls
        
        if not class_maps:
            print(f"      [SKIP] Data for {mode_label} is missing.")
            return

        for region, polygon_coords in self.bappenas_polygons.items():
            for class_id, class_name in self.kelas_dict.items():
                row_dict = {'Wilayah': region, 'Kelas': class_name}
                
                for scen in self.scenarios:
                    da_cls = class_maps.get(scen)
                    if da_cls is None:
                        row_dict[scen] = np.nan
                        continue
                        
                    da_mask = self._get_polygon_mask(da_cls, polygon_coords)
                    da_region = da_cls.where(da_mask)
                    
                    vals = da_region.values
                    total_land_pixels = np.count_nonzero(~np.isnan(vals))
                    
                    if total_land_pixels == 0:
                        pct = 0.0
                    else:
                        class_pixels = np.count_nonzero(vals == class_id)
                        pct = (class_pixels / total_land_pixels) * 100.0
                    
                    row_dict[scen] = round(pct, 1)
                    
                table_data.append(row_dict)

        columns_order = ['Wilayah', 'Kelas'] + self.scenarios
        df = pd.DataFrame(table_data)[columns_order]
        
        out_xlsx = os.path.join(self.tables_dir, f"TABEL_AREA_BAPPENAS_KMEANS{mode}.xlsx")
        df.to_excel(out_xlsx, index=False)
        print(f"      [SAVED] Table exported: {os.path.basename(out_xlsx)}")

    # =========================================================================
    # BLOCK 3: MAIN EXECUTION
    # =========================================================================
    def run_all(self):
        print("==========================================================")
        print("INITIALIZING K-MEANS VISUALIZER & TABLE EXTRACTOR")
        print("==========================================================")
        
        for mode in ['', '_siang']:
            self.plot_kmeans_climatology(mode)
            self.plot_kmeans_seasonal(mode)
            self.extract_bappenas_tables(mode)
            print("-" * 58)
            
        print("\n[SUCCESS] All K-Means maps and Bappenas tables successfully generated!")

if __name__ == "__main__":
    SovereignKMeansVisualizer().run_all()